In [1]:
inicio = '2026.09.01'
fim = ''
day_minus = 0
type = 'rt'


import glob
import pandas as pd
import warnings
import os
import functions
from datetime import datetime
warnings.filterwarnings(
"ignore",
message="DataFrame is highly fragmented",
category=pd.errors.PerformanceWarning
)

dominio_interno = os.getlogin()
logger = functions.logger
print(f'parametros recebidos: inicio {inicio} e  dminus {day_minus}')
logger.write( etapa='Charge', mensagem=f'parametros recebidos: inicio {inicio} e  dminus {day_minus}') 

logger.write( etapa='Charge', mensagem=' carregando dados QMOB') 
# Carregamento das bases do QMOB no nivel de relatório (SURVEI & REPORT ID)
list_df, init = functions.df_charge(type,inicio, fim)

# Carregamento da base do masterfile fixa
logger.write( etapa='Charge', mensagem='Carregando MF')
masterfile = functions.masterfile_data(f'C:/Users/{dominio_interno}/Numerator International/BKO - Documents/Report/Elegibilidade OOH/projeto_ooh/datalake/barcodes_ooh/OOH Masterfile - Barcodes Ativos - 2025_07_15 Resumo.CSV')

# Geração de master, o conjunto principal de onde parte as operações. (QMOB X MASTERFILE)
master = functions.conjunto_principal(list_df,masterfile,csv_path=False,reprocessing=False) # Requisição não está respeitando o filtro de data
logger.write( etapa='Charge', mensagem='Dados Principais carregados MF, QMOB e AMOSTRA_VIVA')

parametros recebidos: inicio 2026.09.01 e  dminus 0
[2026-09-28 09:38:02.893175] [INFO] [Charge] parametros recebidos: inicio 2026.09.01 e  dminus 0
[2026-09-28 09:38:02.893175] [INFO] [Charge]  carregando dados QMOB


c:\Users\luiz.farias\OneDrive - Numerator International\Área de Trabalho\Workspace\PROD\wp_central_atm\automacoes\wp_ooh_atm_amostra_viva_ooh\functions.py:41: DtypeWarning: Columns (0: lookupType, 1: pBR_Extra1_;3, 2: pBR_Extra1_;4, 3: pBR_Extra1_;5, 4: pBR_Extra1_;6, 5: pBR_Extra2_;2, 6: pBR_Extra2_;3, 7: pBR_Extra2_;4, 8: pBR_Extra2_;5, 9: pBR_Extra2_;6, 10: pBR_Extra2_;7, 11: pBR_Extra2_;8, 12: pBR_Extra2_;9, 13: pBR_Extra2_;10, 14: pBR_Extra2_;11, 15: pBR_Extra2_;12, 16: pBR_Extra3_;1, 17: pBR_Extra3_;2, 18: pBR_Extra3_;3, 19: pBR_Extra3_;4, 20: pBR_Extra3_;5, 21: pBR_Extra3_;7, 22: pBR_Extra3_;8, 23: pBR_Extra3_;9, 24: pBR_Extra3_;10, 25: pBR_Extra3_;11, 26: pBR_Extra3_;12, 27: pBR_Extra4_;2, 28: pBR_Extra4_;3, 29: pBR_Extra4_;4, 30: pBR_Extra4_;5, 31: pBR_Extra4_;7, 32: pBR_Extra4_;8, 33: pBR_Extra4_;9, 34: pBR_Extra4_;10, 35: pBR_Extra4_;11, 36: pBR_Extra4_;12, 37: pBR_Extra5_;1, 38: pBR_Extra5_;2, 39: pBR_Extra5_;3, 40: pBR_Extra5_;4, 41: pBR_Extra5_;7, 42: pBR_Extra5_;8, 43: p

[2026-09-28 09:38:02.893175] [INFO] [Charge] Carregando MF
[2026-09-28 09:38:02.893175] [INFO] [Charge] Dados Principais carregados MF, QMOB e AMOSTRA_VIVA


In [ ]:
def transform_w(df):

    df = df.copy()
    import pandas as pd

    aux = {
        'Entry ID': 'EntryID',
        'Entry Type': 'EntryType',
        'Start time': 'StartTime'
    }

    df.rename(columns=aux, inplace=True)

    # Retirar os usuários de teste
    df = df[~df['Username'].isin(['brhansel', '55000016'])]

    df["reportId"] = df["reportId"].astype(str)

    # Série temporária com dtype datetime
    start_time = pd.Series(
        pd.NaT,
        index=df.index,
        dtype="datetime64[ns]"
    )

    # Reporte antigo
    mask_antigo = df["reportId"].eq("78932")

    start_time.loc[mask_antigo] = pd.to_datetime(
        df.loc[mask_antigo, "StartTime"],
        format="%d.%m.%Y %H:%M:%S"
    )

    # Reporte novo
    mask_novo = df["reportId"].eq("135062")

    start_time.loc[mask_novo] = pd.to_datetime(
        df.loc[mask_novo, "StartTime"],
        format="%m.%d.%Y %H:%M:%S"
    )

    # Padroniza tudo
    df["StartTime"] = start_time.dt.date

    df["WeekNumber"] = df["StartTime"].apply(
        lambda x: str(x.isocalendar()[1])
    )

    df["ATO"] = None
    df["DNC"] = None

    df["StartTime"] = start_time.dt.strftime("%Y.%m.%d")

    return df

In [21]:
 # A API não respeita os limites de data, então preciso fazer um filtro no pós-normalização
# Transformação, ajuste de coluna, tipagem de dados, e criação de colunas auxiliares;
master_transformed = transform_w(master)

# Segmentação dos dados por data do mês de referência
logger.write( etapa='Operações', mensagem='Filtragem de dados brutos')
init = datetime.strptime(init, "%Y.%m.%d %H:%M:%S").date()

#logger.write( etapa='info', mensagem=f'Priodo entre: {master_transformed.StartTime} e {master_transformed.StartTime.max() - pd.Timedelta(days=day_minus)}')   
master_transformed = master_transformed.loc[(master_transformed.StartTime >= init) &   
(master_transformed.StartTime <= master_transformed.StartTime.max() - pd.Timedelta(days=day_minus))]


[2026-09-28 09:38:02.893175] [INFO] [Operações] Filtragem de dados brutos
